# OpenAI 매개변수

# 개요
OpenAI 모델에 요청을 보낼 때 여러 매개변수를 사용하여 모델의 동작과 출력을 제어할 수 있습니다. \
이러한 매개변수를 이해하면 텍스트 생성, 질문 응답 또는 기타 사용 사례에 맞게 응답을 세부 조정할 수 있습니다.

더 자세한 예제는 공식 문서를 참조하세요: [Azure OpenAI Service](https://learn.microsoft.com/en-us/azure/ai-services/openai/reference)


In [19]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

# 환경 변수 로드
dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
azure_openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview")
CHAT_COMPLETIONS_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5.4-mini")

if not azure_openai_endpoint or not azure_openai_key:
    raise ValueError(".env 파일에 AZURE_OPENAI_ENDPOINT와 AZURE_OPENAI_KEY를 설정하세요.")

client = AzureOpenAI(
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_key,
    api_version=azure_openai_api_version,
    default_headers={"Ocp-Apim-Subscription-Key": azure_openai_key},
)

SEED = 123

print(f"Azure OpenAI endpoint: {azure_openai_endpoint}")
print(f"API version: {azure_openai_api_version}")
print(f"Chat deployment: {CHAT_COMPLETIONS_MODEL}")

Azure OpenAI endpoint: https://apim-ai-workshop-010.azure-api.net/
API version: 2025-04-01-preview
Chat deployment: gpt-5.4-mini


# 매개변수: max_completion_tokens

**설명**: 생성할 응답의 최대 토큰 수를 설정합니다.  
**기본값**: 모델과 API 설정에 따라 달라질 수 있습니다.  
**예제**: `max_completion_tokens=50`

`max_completion_tokens`는 모델이 생성할 수 있는 출력 토큰의 상한을 정합니다. 값이 작으면 답변이 짧게 잘릴 수 있고, 값이 크면 더 긴 답변을 생성할 수 있습니다.

이전 예제나 일부 문서에서는 `max_tokens`라는 이름을 볼 수 있지만, 이 워크숍의 Azure OpenAI Chat Completions 예제에서는 `max_completion_tokens`를 사용합니다.

In [20]:
def call_openai_with_max_completion_tokens(max_completion_tokens):
    response = client.chat.completions.create(
        model=CHAT_COMPLETIONS_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "최고의 반려동물은 "},
        ],
        max_completion_tokens=max_completion_tokens,
    )
    return response.choices[0].message.content

completion_tokens = [32, 64, 120, 200]
for token in completion_tokens:
    print(f"Max Completion Tokens: {token}\n")
    print(call_openai_with_max_completion_tokens(token))
    print("\n" + "-" * 80 + "\n")

Max Completion Tokens: 32

“최고의 반려동물”은 사람마다 달라요.  
원하는 생활 방식에 따라 달라집니다:

-

--------------------------------------------------------------------------------

Max Completion Tokens: 64

“최고의 반려동물은”은 사람마다 달라요.  
보통은 이런 기준으로 많이 골라요:

- **강아지**: 교감이 깊고 활동적
- **고양이**: 비교적 독립적이고 관리가 편함

--------------------------------------------------------------------------------

Max Completion Tokens: 120

“최고의 반려동물”은 사람마다 달라요.  
보통은 **생활환경, 시간, 예산, 성격**에 따라 달라집니다.

간단히 예를 들면:
- **강아지**: 교감이 깊고 활동적
- **고양이**: 비교적 독립적이고 관리가 편한 편
- **햄스터/토끼/물고기**: 공간이 적게 들고 비교적 조용함

원하시면 제가  
**“혼자 사

--------------------------------------------------------------------------------

Max Completion Tokens: 200

“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 알레르기, 성격에 따라 다르지만 보통 많이 추천되는 건:

- **강아지**: 교감이 크고 충성심이 높음
- **고양이**: 비교적 독립적이고 관리가 편한 편
- **햄스터/토끼/물고기**: 공간이 적게 들고 비교적 조용함

원하시면 **혼자 사는 사람에게 좋은 반려동물**, **아이 있는 집**, **아파트에 적합한 반려동물**처럼 상황별로 추천해드릴게요.

--------------------------------------------------------------------------------


# 매개변수: temperature

**설명**: 출력의 무작위성을 제어합니다. 낮은 값은 출력을 더 결정론적으로 만들고, 높은 값은 무작위성을 증가시킵니다. \
모델이 다음 토큰을 선택할 때, 여러 후보 중에서
- 높은 확률 토큰을 쓸지
- 조금 낮은 확률 토큰까지 선택할 여지를 줄지

**값 범위**: 0에서 2 \
**기본값**: 1 \
**예제**: temperature=0.7

높은 값은 모델이 더 창의적인 출력을 생성하도록 합니다.

---
**참고**: 일반적으로 이 매개변수 또는 top_p를 조정하되 둘 다 동시에 조정하지 않는 것을 권장합니다.


In [ ]:
def call_openai(num_times, prompt, temperature=0.7, use_seed=False):
    for _ in range(num_times):
        request = {
            "model": CHAT_COMPLETIONS_MODEL,
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ],
            "max_completion_tokens": 200,
            "temperature": temperature,
        }

        if use_seed:
            request["seed"] = SEED

        response = client.chat.completions.create(**request)
        print(response.choices[0].message.content)
        print("-" * 80)

In [ ]:
# Without seed and temperature, the response is different each time
call_openai(10, '최고의 반려동물은 ')

“최고의 반려동물은”은 사람마다 달라요.  
성격, 생활환경, 시간, 예산에 따라 달라지거든요.

간단히 보면:

- **강아지**: 교감이 깊고 활동적

--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 성격에 따라 달라지기 때문이죠.

간단히 말하면:

- **처음 키우기 쉬운 편**: 고양
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활패턴, 공간, 예산, 알레르기, 돌볼 수 있는 시간에 따라 달라집니다.

간단히 추천하면:

- **강아지**: 교
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활패턴, 공간, 예산, 알레르기, 돌볼 수 있는 시간에 따라 달라집니다.

간단히 말하면:
- **강아지**: 교
--------------------------------------------------------------------------------
“최고의 반려동물은”은 사람마다 달라요.  
보통은 **내 생활패턴, 공간, 시간, 예산, 알레르기**에 가장 잘 맞는 동물이 최고의 반려동물입니다.

예를 들면
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 알레르기, 성격에 따라 다르거든요.

간단히 추천하면:

- **강아지**: 교감이 크
-------------------------------------------------------------

In [23]:
# Now using a seed and 0 temperature, the response is much more consistent
call_openai(10, '최고의 반려동물은 ', temperature=0, use_seed=True)

“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 알레르기, 성격에 따라 달라집니다.

간단히 추천하면:
- **강아지**: 교감이 크고
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 알레르기, 성격에 따라 달라집니다.

간단히 추천하면:
- **강아지**: 교감이 크고
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 알레르기, 성격에 따라 달라집니다.

간단히 추천하면:
- **강아지**: 교감이 크고
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 알레르기, 성격에 따라 달라집니다.

간단히 추천하면:
- **강아지**: 교감이 크고
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 알레르기, 성격에 따라 달라집니다.

간단히 추천하면:
- **강아지**: 교감이 크고
--------------------------------------------------------------------------------
“최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 알레르기, 성격에 따라 달라집니다.

간단히 추천하면:
- **강아지**: 교감이 크고
-------------------------------------------------------------------------

# 매개변수: n

**설명**: 각 프롬프트에 대해 생성할 응답의 수를 지정합니다.  
**기본값**: 1  
**예제**: `n=3`

---
**참고**: 이 매개변수는 여러 응답을 생성하므로 토큰 할당량을 빠르게 소모할 수 있습니다. 신중하게 사용하고 `max_completion_tokens` 및 stop 설정이 적절한지 확인하세요.

In [24]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "최고의 반려동물은"},
    ],
    max_completion_tokens=60,
    n=3,
)

for index, choice in enumerate(response.choices):
    print(index, choice.message.content)
    print("\n" + "-" * 80 + "\n")

0 “최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 예산, 성격에 따라 달라집니다.

간단히 말하면:

- **강아지**: 애정 표현이 크고 함께하는 재미가

--------------------------------------------------------------------------------

1 “최고의 반려동물”은 사람마다 다르지만, 보통은 **당신의 생활방식과 잘 맞는 동물**이 가장 좋은 반려동물이에요.

예를 들면:
- **강아지**: 교감

--------------------------------------------------------------------------------

2 “최고의 반려동물”은 사람마다 달라요.  
생활환경, 시간, 알레르기, 성격, 예산에 따라 달라지거든요.

간단히 말하면:

- **강아지**: 가장 친밀

--------------------------------------------------------------------------------



# 매개변수: presence_penalty
**설명**: 텍스트에 이미 나타난 토큰을 기반으로 새 토큰에 페널티를 부여하여 모델이 새로운 토큰을 사용하도록 유도합니다.  
**값 범위**: -2.0에서 2.0  
**기본값**: 0  
**예제**: `presence_penalty=0.5`

**특징**  
- 단어가 *한 번이라도 등장*했으면 그 이후 사용 확률을 낮춤  
- "존재 여부" 기반이므로 반복 횟수와는 무관  
- 같은 주제 안에서 새로운 소재, 관점, 아이디어로 이동하게 만들 때 적합  
- 반복을 금지하는 기능은 아니며 효과가 비교적 미묘할 수 있음

**주의 사항**  
- 코드 생성·SQL·JSON·Tool Call처럼 **반복이 필요한 출력에서는 권장하지 않음**  
- 주제어가 반복될 수밖에 없는 설명형 프롬프트에서는 효과가 제한적  
- 값이 너무 높으면 문맥과 자연스러움이 손상될 수 있음

---

### presence_penalty vs frequency_penalty 비교

| 특징 | presence_penalty | frequency_penalty |
|------|------------------|-------------------|
| **기준** | 존재 여부 (0/1) | 반복 횟수 (누적) |
| **강도** | 약함 | 강함 |
| **효과** | 새로운 소재나 관점으로 이동 | 반복 단어와 반복 문장 구조 억제 |
| **용도** | 아이디어 다양화 | 반복 패턴 제거 |
| **예시** | 반려동물 제목 소재 다양화 | 같은 표현 반복 줄이기 |

아래 예제는 차이를 비교하기 쉽도록 `고양이`보다 넓은 `반려동물` 주제로 제목만 짧게 출력합니다. `temperature`와 `seed`를 고정하고 `presence_penalty`만 바꾸어 동물, 상황, 관점이 얼마나 다양해지는지 관찰합니다.

In [25]:
def call_openai_with_presence_penalty(presence_penalty):
    response = client.chat.completions.create(
        model=CHAT_COMPLETIONS_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": """
반려동물을 주제로 블로그 제목 12개를 만들어줘.
제목만 번호 목록으로 출력하고, 설명 문장은 쓰지 마.
각 제목은 20자 이내로 짧게 작성해줘.
""",
            },
        ],
        max_completion_tokens=300,
        temperature=0.8,
        seed=SEED,
        presence_penalty=presence_penalty,
    )
    return response.choices[0].message.content

# Generate with different presence_penalty values
penalties = [0, 0.5, 1.0, 1.5]
for penalty in penalties:
    print(f"Presence Penalty: {penalty}\n")
    print(call_openai_with_presence_penalty(penalty))
    print("\n" + "-" * 80 + "\n")

Presence Penalty: 0

1. 반려동물과의 하루  
2. 우리집 강아지 이야기  
3. 고양이와 사는 즐거움  
4. 반려동물 입양기  
5. 사랑스러운 반려친구  
6. 펫과 함께하는 삶  
7. 강아지 산책 일기  
8. 고양이 집사의 기록  
9. 반려동물 꿀팁 모음  
10. 우리집 펫 성장기  
11. 동물과 마음 나누기  
12. 반려생활 소소한 행복

--------------------------------------------------------------------------------

Presence Penalty: 0.5

1. 반려동물과의 하루  
2. 우리집 강아지 이야기  
3. 고양이와 사는 즐거움  
4. 반려동물 입양 준비  
5. 펫과 함께하는 산책  
6. 반려동물 건강관리 팁  
7. 강아지 훈련 시작하기  
8. 고양이 집사 일상  
9. 반려동물 간식 고르기  
10. 펫 용품 추천 모음  
11. 사랑스런 반려동물 기록  
12. 반려동물과 여행하기

--------------------------------------------------------------------------------

Presence Penalty: 1.0

1. 반려동물과의 하루  
2. 우리집 강아지 이야기  
3. 고양이와 사는 즐거움  
4. 반려동물 입양 준비  
5. 펫과 함께하는 산책  
6. 반려동물 건강관리 팁  
7. 강아지 훈련 시작하기  
8. 고양이 집사 일상  
9. 반려동물 간식 추천  
10. 소중한 나의 펫  
11. 펫과 떠나는 여행  
12. 반려동물 행복 노트

--------------------------------------------------------------------------------

Presence Penalty: 1.5

1. 반려동물과의 하루  
2. 우리집 강아지 이야기  
3. 고양이와 사는 즐거움  
4. 반려동물 입양기  
5. 사랑스러운 털친구들  
6.

# 매개변수: frequency_penalty
**설명**: 텍스트에 이미 나타난 빈도를 기반으로 새 토큰에 페널티를 부여하여 같은 단어나 표현을 반복할 가능성을 줄입니다.  
**값 범위**: -2.0에서 2.0  
**기본값**: 0  
**예제**: `frequency_penalty=0.5`

**특징**  
- 동일 단어가 반복될수록 페널티 강도가 증가  
- presence_penalty보다 반복 억제 효과가 더 직접적임  
- 반복되는 단어, 문장 시작, 문장 패턴을 줄이고 싶을 때 적합  
- 창의적 글쓰기, 마케팅 문구, 스토리텔링에 유용

**주의 사항**  
- 변수명·키 이름·구문 반복이 중요한 **코드/SQL/JSON 생성에는 권장하지 않음**  
- 높은 값일수록 다양성은 증가하지만 문맥 일관성은 낮아질 수 있음  
- 값이 너무 높으면 반복을 피하려다 사용자의 형식 지시를 어길 수 있음  
- 정보 전달이나 기술적 설명 같은 정확성이 필요한 응답에는 추천하지 않음

아래 예제는 고양이를 소개하는 짧은 문장 여러 개를 요청합니다. `temperature`와 `seed`를 고정하고 `frequency_penalty`만 바꾸어 같은 단어와 표현 반복을 피하려는 경향을 관찰합니다.

In [26]:
def call_openai_with_frequency_penalty(frequency_penalty):
    response = client.chat.completions.create(
        model=CHAT_COMPLETIONS_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": """
고양이를 소개하는 짧은 문장 12개를 작성해줘.
각 문장은 15자 이내로 작성해줘.
제목이나 설명 없이 번호 목록만 출력해줘.
""",
            },
        ],
        max_completion_tokens=300,
        temperature=0.8,
        seed=SEED,
        frequency_penalty=frequency_penalty,
    )
    return response.choices[0].message.content

# Generate with different frequency_penalty values
penalties = [0, 0.5, 1.0]
for penalty in penalties:
    print(f"Frequency Penalty: {penalty}\n")
    print(call_openai_with_frequency_penalty(penalty))
    print("\n" + "-" * 80 + "\n")

Frequency Penalty: 0

1. 고양이는 귀엽다.
2. 고양이는 우아하다.
3. 고양이는 호기심 많다.
4. 고양이는 잠을 좋아해.
5. 고양이는 조용하다.
6. 고양이는 장난꾸러기.
7. 고양이는 따뜻하다.
8. 고양이는 매력적이다.
9. 고양이는 독립적이다.
10. 고양이는 사랑스럽다.
11. 고양이는 민첩하다.
12. 고양이는 포근하다.

--------------------------------------------------------------------------------

Frequency Penalty: 0.5

1. 고양이는 귀엽다.
2. 고양이는 우아하다.
3. 고양이는 호기심 많다.
4. 고양이는 잠을 좋아해.
5. 고양이는 조용하다.
6. 고양이는 장난꾸러기다.
7. 고양이는 포근하다.
8. 고양이는 사랑스럽다.
9. 고양이는 독립적이다.
10. 고양이는 발이 가볍다.
11. 고양이는 눈빛이 맑다.
12. 고양이는 늘 매력적이다.

--------------------------------------------------------------------------------

Frequency Penalty: 1.0

1. 고양이는 귀엽다.
2. 고양이는 우아하다.
3. 고양이는 호기심 많다.
4. 고양이는 잠을 좋아해.
5. 고양이는 조용하다.
6. 고양이는 장난꾸러기다.
7. 고양이는 포근하다.
8. 고양이는 사랑스럽다.
9. 고양이는 독립적이다.
10. 고양이는 발이 가볍다.
11. 고양이는 눈빛이 맑다.
12. 고양이는 늘 매력적이다。

--------------------------------------------------------------------------------



### 탐색할 사용 사례
1. **응답 비교**  
   여러 응답을 생성하여 사용 사례에 가장 적합한 결과를 선택하세요.

2. **다양성 증가**  
   창의적 응용 프로그램에서 다양한 표현을 얻기 위해 여러 응답을 생성하세요.

3. **강건성 향상**  
   여러 응답을 생성하여 일관성과 정확성을 비교·검증하세요.

---

### 모범 사례
1. **프롬프트 길이 최적화**  
   프롬프트는 간결하지만 충분한 정보를 담도록 작성하세요.

2. **Temperature 및 Top_p 조정**  
   결정론적 vs. 창의적 응답의 균형을 위해 온도를 적절히 조정하세요.

3. **토큰 사용량 모니터링**  
   `max_completion_tokens`를 적절히 설정하여 비용과 응답 길이를 관리하세요.

4. **중지 시퀀스 사용**  
   모델이 텍스트 생성을 중단해야 하는 지점을 정의해 출력을 안정적으로 제어하세요.

5. **여러 응답 생성**  
   `n` 매개변수를 사용하여 여러 응답을 생성하고 필요한 응답을 선택하세요.